# VoltRelay Energy — Operations Analytics
## Gradient Learnings Data Analytics Hackathon

**Project:** Battery Swapping Network Operations Analysis  
**Analysis period:** January 2024 – June 2025  
**Analyst:** Nishant Ganesh Rathod

### Business objective
Understand network performance, service reliability, battery health, pricing patterns and rider-experience risks across a six-city battery-swapping network.

### Dataset inventory

| Dataset | Rows | Purpose |
|---|---:|---|
| `swap_events.csv` | 3,877,013 | Individual swap attempts and transaction amounts |
| `station_hourly_status.csv` | 1,487,712 | Hourly station telemetry |
| `riders.csv` | 20,000 | Rider profiles and signup information |
| `batteries.csv` | 6,500 | Battery asset and health records |
| `support_tickets.csv` | 44,000 | Customer support records |
| `stations.csv` | 152 | Station attributes and locations |
| `city_daily_context.csv` | 3,282 | City-level weather and calendar context |
| `fleet_partners.csv` | 12 | Fleet partner and contract information |

**Data note:** All eight datasets are included. `swap_events.csv` is large, so it is processed in chunks rather than loaded fully into memory. Transaction-level results are calculated from swap records; station telemetry and support tickets are analysed separately as operational/customer-experience evidence.

**Interpretation note:** This is descriptive and diagnostic analysis. Support-ticket activity is not the same as confirmed rider churn, and recorded revenue is not profit or contribution margin.


In [ ]:
# ============================================================
# SECTION 1: SETUP & DATA LOADING
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams.update({
    'figure.facecolor': '#1a1d27',
    'axes.facecolor': '#22263a',
    'axes.edgecolor': '#3d4166',
    'axes.labelcolor': '#9ca3c0',
    'xtick.color': '#9ca3c0',
    'ytick.color': '#9ca3c0',
    'text.color': '#e8eaf6',
    'grid.color': '#2d3148',
    'grid.linestyle': '--',
    'grid.alpha': 0.5,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.titleweight': 'bold'
})
ACCENT = ['#4f8ef7','#f7c94f','#4ff7a0','#f74f4f','#f97316','#bf4ff7']

print('Libraries loaded. Begin data loading...')

In [ ]:
# ============================================================
# DATA LOADING — ALL 8 DATASETS
# ============================================================
from pathlib import Path

# Expected project structure:
# project_folder/
#   VoltRelay_Analytics_Notebook_VSCode.ipynb
#   data/
#       riders.csv, stations.csv, ... , swap_events.csv

DATA_DIR = Path.cwd() / "data"

required_files = [
    "riders.csv", "batteries.csv", "stations.csv", "fleet_partners.csv",
    "support_tickets.csv", "city_daily_context.csv",
    "station_hourly_status.csv", "swap_events.csv"
]
missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(
        f"Missing files in {DATA_DIR}: {missing_files}. "
        "Open VS Code in the project root folder containing the data/ directory."
    )

riders = pd.read_csv(DATA_DIR / "riders.csv")
batteries = pd.read_csv(DATA_DIR / "batteries.csv")
stations = pd.read_csv(DATA_DIR / "stations.csv")
fleet = pd.read_csv(DATA_DIR / "fleet_partners.csv")
tickets = pd.read_csv(DATA_DIR / "support_tickets.csv", parse_dates=["created_ts"])
city_ctx = pd.read_csv(DATA_DIR / "city_daily_context.csv", parse_dates=["date"])

print("Loading station_hourly_status.csv...")
shs = pd.concat(
    pd.read_csv(DATA_DIR / "station_hourly_status.csv",
                chunksize=200_000, parse_dates=["hour_start"]),
    ignore_index=True
)

print("\nSeven reference/telemetry datasets loaded:")
for name, df in [
    ("riders", riders), ("batteries", batteries), ("stations", stations),
    ("fleet", fleet), ("tickets", tickets), ("city_ctx", city_ctx), ("shs", shs)
]:
    print(f"  {name}: {df.shape[0]:,} rows × {df.shape[1]} columns")

print("\nEighth dataset found:", (DATA_DIR / "swap_events.csv").name)
print("Size (MB):", round((DATA_DIR / "swap_events.csv").stat().st_size / (1024**2), 2))
print("Swap events will be processed in chunks to avoid loading the full 695 MB file.")


## Swap Events — Transaction-Level Audit and Q1 Metrics

The following cells process all 3,877,013 swap attempts in chunks. They calculate monthly attempts, completions, failure categories and recorded revenue without loading the entire file into memory.


In [ ]:
# ============================================================
# SWAP EVENTS — CHUNKED TRANSACTION AUDIT
# ============================================================
SWAP_PATH = DATA_DIR / "swap_events.csv"
CHUNK_SIZE = 200_000

event_counts = {}
event_monthly_parts = []
event_missing = {}
event_min_date = None
event_max_date = None
event_rows = 0

for chunk in pd.read_csv(SWAP_PATH, chunksize=CHUNK_SIZE, parse_dates=["event_ts"]):
    event_rows += len(chunk)

    # Event-type totals
    counts = chunk["event_type"].value_counts(dropna=False)
    for event_type, count in counts.items():
        event_counts[event_type] = event_counts.get(event_type, 0) + int(count)

    # Date coverage
    cmin, cmax = chunk["event_ts"].min(), chunk["event_ts"].max()
    event_min_date = cmin if event_min_date is None else min(event_min_date, cmin)
    event_max_date = cmax if event_max_date is None else max(event_max_date, cmax)

    # Missing values
    for col, count in chunk.isna().sum().items():
        event_missing[col] = event_missing.get(col, 0) + int(count)

    # Monthly aggregation
    chunk["month"] = chunk["event_ts"].dt.to_period("M").astype(str)
    chunk["is_completed"] = chunk["event_type"].eq("swap_completed")
    chunk["is_no_charged_battery"] = chunk["event_type"].eq("failed_no_charged_battery")
    chunk["is_abandoned_queue"] = chunk["event_type"].eq("abandoned_queue")
    chunk["is_cancelled"] = chunk["event_type"].eq("cancelled_by_rider")
    chunk["is_system_error"] = chunk["event_type"].eq("failed_system_error")

    completed = chunk[chunk["is_completed"]]
    agg = chunk.groupby("month").agg(
        total_attempts=("event_type", "size"),
        completed_swaps=("is_completed", "sum"),
        no_charged_battery=("is_no_charged_battery", "sum"),
        abandoned_queue=("is_abandoned_queue", "sum"),
        rider_cancelled=("is_cancelled", "sum"),
        system_errors=("is_system_error", "sum")
    ).reset_index()

    revenue = completed.groupby("month")["amount_charged_inr"].sum()
    discounts = completed.groupby("month")["discount_inr"].sum()
    agg["recorded_revenue_inr"] = agg["month"].map(revenue).fillna(0)
    agg["discounts_inr"] = agg["month"].map(discounts).fillna(0)
    event_monthly_parts.append(agg)

monthly_swap_metrics = (
    pd.concat(event_monthly_parts, ignore_index=True)
      .groupby("month", as_index=False)
      .sum(numeric_only=True)
      .sort_values("month")
)

monthly_swap_metrics["completion_rate_pct"] = (
    monthly_swap_metrics["completed_swaps"] /
    monthly_swap_metrics["total_attempts"] * 100
)
for col in ["no_charged_battery", "abandoned_queue", "rider_cancelled", "system_errors"]:
    monthly_swap_metrics[col + "_rate_pct"] = (
        monthly_swap_metrics[col] / monthly_swap_metrics["total_attempts"] * 100
    )
monthly_swap_metrics["avg_revenue_per_completed_swap_inr"] = (
    monthly_swap_metrics["recorded_revenue_inr"] /
    monthly_swap_metrics["completed_swaps"].replace(0, np.nan)
)

print(f"Rows processed: {event_rows:,}")
print("Date range:", event_min_date, "to", event_max_date)
print("\nEvent counts:")
display(pd.Series(event_counts, name="count").to_frame())
print("\nMissing values (non-zero only):")
display(pd.Series({k:v for k,v in event_missing.items() if v > 0}, name="missing_count").to_frame())
print("\nMonthly transaction metrics:")
display(monthly_swap_metrics.round(2))


In [ ]:
# ============================================================
# Q1 — VERIFIED OVERALL TRANSACTION KPIs
# ============================================================
total_attempts = int(monthly_swap_metrics["total_attempts"].sum())
total_completed = int(monthly_swap_metrics["completed_swaps"].sum())
total_recorded_revenue = monthly_swap_metrics["recorded_revenue_inr"].sum()
total_discounts = monthly_swap_metrics["discounts_inr"].sum()

print(f"Total swap attempts: {total_attempts:,}")
print(f"Completed swaps: {total_completed:,}")
print(f"Overall completion rate: {total_completed / total_attempts * 100:.2f}%")
print(f"Recorded revenue on completed swaps: ₹{total_recorded_revenue:,.2f}")
print(f"Recorded discounts on completed swaps: ₹{total_discounts:,.2f}")
print(f"Average recorded revenue per completed swap: ₹{total_recorded_revenue / total_completed:.2f}")

# This is recorded revenue, not profit or contribution margin.


In [ ]:
# ============================================================
# SECTION 2: DATA CLEANING
# ============================================================

# 2.1 City name standardization (riders.home_city)
print('Unique home_city values before cleaning:', riders['home_city'].nunique())
city_map = {
    'Bengaluru':'Bengaluru','BLR':'Bengaluru','bengaluru':'Bengaluru','Bangalore':'Bengaluru',
    'Delhi NCR':'Delhi NCR','New Delhi':'Delhi NCR','Delhi':'Delhi NCR','Gurgaon':'Delhi NCR',
    'Hyderabad':'Hyderabad','HYD':'Hyderabad','Hyd':'Hyderabad','hyderabad':'Hyderabad',
    'Pune':'Pune','PUN':'Pune','pune':'Pune',
    'Mumbai':'Mumbai','MUM':'Mumbai','Bombay':'Mumbai',
    'Jaipur':'Jaipur','JAI':'Jaipur','jaipur':'Jaipur'
}
riders['city'] = riders['home_city'].map(city_map)
print('Unique cities after cleaning:', riders['city'].nunique())
print(riders['city'].value_counts())

# 2.2 Date parsing
stations['commissioned_date'] = pd.to_datetime(stations['commissioned_date'])
batteries['commission_date'] = pd.to_datetime(batteries['commission_date'])
batteries['manufacture_date'] = pd.to_datetime(batteries['manufacture_date'])
riders['signup_date'] = pd.to_datetime(riders['signup_date'])

# 2.3 Derived columns - station hourly
shs['stockout_2w'] = shs['charged_2w_min'] == 0
shs['month'] = shs['hour_start'].dt.to_period('M')
shs['hour'] = shs['hour_start'].dt.hour

# 2.4 Merge station attributes into hourly
shs = shs.merge(stations[['station_id','city','expansion_wave','charger_generation',
                            'location_type','connectivity_tier']], on='station_id', how='left')

# 2.5 Battery degradation
batteries['soh_degradation'] = batteries['initial_soh_pct'] - batteries['current_soh_pct']
batteries['age_months'] = (pd.Timestamp('2025-06-30') - batteries['commission_date']).dt.days / 30

# 2.6 Tickets enrichment
tickets = tickets.merge(stations[['station_id','city','expansion_wave','charger_generation',
                                    'location_type','connectivity_tier']], on='station_id', how='left')
tickets['month'] = tickets['created_ts'].dt.to_period('M')
tickets['hour'] = tickets['created_ts'].dt.hour

print('\nData cleaning complete.')
print(f"Stockout column added. Overall stockout rate: {shs['stockout_2w'].mean()*100:.2f}%")

In [ ]:
# ============================================================
# Q1: NETWORK PERFORMANCE OVER TIME
# ============================================================

ok = shs[shs['telemetry_status']=='ok']
monthly_shs = ok.groupby('month').agg(
    total_hours=('station_id','count'),
    stockout_hours=('stockout_2w','sum'),
    avg_charge=('avg_charge_minutes','mean'),
    outage_min=('outage_minutes','mean'),
    quarantined=('packs_quarantined','mean')
).reset_index()
monthly_shs['stockout_rate_pct'] = monthly_shs['stockout_hours'] / monthly_shs['total_hours'] * 100
monthly_shs = monthly_shs[monthly_shs['month'] >= '2024-01']

monthly_tickets = tickets.groupby('month').size().reset_index(name='tickets')
monthly_tickets = monthly_tickets[(monthly_tickets['month'] >= '2024-01') & 
                                    (monthly_tickets['month'] <= '2025-06')]

riders['signup_month'] = riders['signup_date'].dt.to_period('M')
monthly_signups = riders.groupby('signup_month').size().reset_index(name='signups')
monthly_signups = monthly_signups[(monthly_signups['signup_month'] >= '2024-01') & 
                                    (monthly_signups['signup_month'] <= '2025-06')]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Q1: Network Performance Metrics Over Time (Jan 2024 – Jun 2025)', 
             fontsize=15, fontweight='bold', color='#e8eaf6', y=1.01)

months_str = [str(m) for m in monthly_shs['month']]
x = range(len(months_str))

# Stockout rate
ax = axes[0,0]
bars = ax.bar(x, monthly_shs['stockout_rate_pct'], color=ACCENT[3], alpha=0.8)
ax.set_title('2W Stockout Rate (% hours with zero charged stock)')
ax.set_xticks(x[::3]); ax.set_xticklabels(months_str[::3], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('% hours'); ax.yaxis.set_major_formatter(mtick.PercentFormatter())
# Annotate May spikes
ax.annotate('May\nheat\ncrisis', xy=(4, 7.3), xytext=(6, 6.5), fontsize=9, color='#f7c94f',
            arrowprops=dict(arrowstyle='->', color='#f7c94f'))
ax.grid(True, axis='y')

# Ticket volume
ax = axes[0,1]
mt_x = range(len(monthly_tickets))
ax.bar(mt_x, monthly_tickets['tickets'], color=ACCENT[0], alpha=0.8)
ax.set_title('Monthly Support Ticket Volume')
mt_labs = [str(m) for m in monthly_tickets['month']]
ax.set_xticks(list(mt_x)[::3]); ax.set_xticklabels(mt_labs[::3], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Tickets'); ax.grid(True, axis='y')
ax.annotate(f"+718%\nJan→Jun", xy=(17, 5438), xytext=(14, 4800), fontsize=9, color='#f74f4f',
            arrowprops=dict(arrowstyle='->', color='#f74f4f'))

# Rider signups
ax = axes[1,0]
su_x = range(len(monthly_signups))
ax.plot([str(m) for m in monthly_signups['signup_month']], monthly_signups['signups'], 
        color=ACCENT[2], linewidth=2, marker='o', markersize=4)
ax.set_title('New Rider Sign-ups per Month')
ax.set_xticklabels([str(m) for m in monthly_signups['signup_month']], rotation=45, ha='right', fontsize=9)
ax.set_xticks(range(0, len(monthly_signups), 3))
ax.set_xticklabels([str(monthly_signups['signup_month'].iloc[i]) for i in range(0, len(monthly_signups), 3)], 
                    rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Signups'); ax.grid(True)

# Average charge time
ax = axes[1,1]
ax.plot(x, monthly_shs['avg_charge'], color=ACCENT[1], linewidth=2, marker='o', markersize=4)
ax.set_title('Average Charge Time per Month')
ax.set_xticks(x[::3]); ax.set_xticklabels(months_str[::3], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Minutes'); ax.grid(True)
ax.axhline(y=90, color='#f74f4f', linestyle='--', alpha=0.5, label='90 min threshold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/q1_network_performance.png', dpi=150, bbox_inches='tight',
            facecolor='#1a1d27')
plt.show()

print('Interpret ticket volume alongside rider growth; ticket counts alone do not establish churn.')
print('May stockout peaks are descriptive patterns; heat may be a contributing factor and is not proven as the sole cause.')

In [ ]:
# ============================================================
# Q2: SERVICE FAILURES & CUSTOMER EXPERIENCE
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Q2: Service Failures & Customer Experience', fontsize=15, fontweight='bold', 
             color='#e8eaf6', y=1.02)

# Hourly stockout rate
ax = axes[0]
hourly_stockout = ok.groupby('hour')['stockout_2w'].mean() * 100
colors = ['#f74f4f' if v > 2 else '#f97316' if v > 0.5 else '#4f8ef7' for v in hourly_stockout]
ax.bar(hourly_stockout.index, hourly_stockout.values, color=colors)
ax.set_title('Stockout Rate by Hour of Day')
ax.set_xlabel('Hour (0–23)'); ax.set_ylabel('% hours with zero 2W stock')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.axvspan(12, 17, alpha=0.1, color='red', label='Danger zone 12–5pm')
ax.legend(fontsize=9); ax.grid(True, axis='y')

# Ticket categories
ax = axes[1]
cat_counts = tickets['category'].value_counts()
wedges, texts, autotexts = ax.pie(cat_counts.values, labels=cat_counts.index,
    autopct='%1.1f%%', colors=ACCENT, startangle=90,
    textprops={'color':'#e8eaf6','fontsize':9})
ax.set_title('Support Tickets by Category')

# City stockout
ax = axes[2]
city_so = ok.groupby('city')['stockout_2w'].mean() * 100
city_so_sorted = city_so.sort_values(ascending=True)
bar_colors = ['#f74f4f' if v > 2 else '#f97316' if v > 1 else '#4ff7a0' for v in city_so_sorted]
ax.barh(city_so_sorted.index, city_so_sorted.values, color=bar_colors)
ax.set_title('Annual Stockout Rate by City')
ax.set_xlabel('% hours with zero 2W stock')
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.grid(True, axis='x')

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/q2_service_failures.png', dpi=150, bbox_inches='tight',
            facecolor='#1a1d27')
plt.show()

print('Hourly stockout chart shows the observed time-of-day pattern in valid telemetry.')
print('Jaipur (3.21%) and Delhi NCR (2.23%) have 5–10× higher stockout rates than Mumbai/Pune.')
print('No battery available is the #1 complaint at 26.5% of all tickets.')

In [ ]:
# ============================================================
# Q3: STATION & GEOGRAPHIC PATTERNS
# ============================================================

gen_perf = ok.groupby('charger_generation').agg(
    stockout_rate=('stockout_2w','mean'),
    charge_time=('avg_charge_minutes','mean'),
    total_hours=('station_id','count')
).reset_index()

# Worst stations
station_perf = ok.groupby('station_id').agg(
    stockout_rate=('stockout_2w','mean'),
    avg_charge=('avg_charge_minutes','mean'),
    quarantine=('packs_quarantined','mean'),
    obs=('station_id','count')
).reset_index()
worst15 = station_perf.nlargest(15,'stockout_rate')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Q3: Station & Geographic Performance', fontsize=15, fontweight='bold',
             color='#e8eaf6', y=1.02)

# Charge time by gen
ax = axes[0]
gens = gen_perf['charger_generation'].tolist()
charge_vals = gen_perf['charge_time'].tolist()
stockout_vals = [v*100 for v in gen_perf['stockout_rate'].tolist()]
x = np.arange(len(gens))
bars = ax.bar(x, charge_vals, color=ACCENT[:3], alpha=0.85, width=0.4, label='Charge Time (min)')
ax2 = ax.twinx()
ax2.plot(x, stockout_vals, 'o--', color='#f74f4f', linewidth=2, markersize=8, label='Stockout %')
ax.set_xticks(x); ax.set_xticklabels(gens)
ax.set_ylabel('Avg Charge Time (min)', color='#9ca3c0')
ax2.set_ylabel('Stockout Rate %', color='#f74f4f')
ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title('Charger Generation: Charge Time vs Stockout Rate')
# Add value labels
for bar, v in zip(bars, charge_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{v:.0f}m', 
            ha='center', va='bottom', fontsize=10, fontweight='bold', color='#e8eaf6')
ax.grid(True, axis='y', alpha=0.3)

# Worst 15 stations
ax = axes[1]
colors_15 = ['#f74f4f' if 'JAI' in s else '#f97316' if 'DEL' in s else '#f7c94f' 
             for s in worst15['station_id']]
ax.barh(worst15['station_id'], worst15['stockout_rate']*100, color=colors_15)
ax.set_title('15 Worst Stations by Stockout Rate')
ax.set_xlabel('% hours with zero 2W stock')
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
# Legend
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#f74f4f',label='Jaipur'),Patch(color='#f97316',label='Delhi NCR'),
                    Patch(color='#f7c94f',label='Other')], fontsize=9, loc='lower right')
ax.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/q3_station_patterns.png', dpi=150, bbox_inches='tight',
            facecolor='#1a1d27')
plt.show()

print('KEY FINDING: Gen1 charge time 91.8 min vs Gen3 42.1 min — a 2.2× throughput gap.')
print('Gen1 stockout rate 2.33% vs Gen3 0.27% — an 8.7× difference.')
print('The station chart identifies the 15 highest observed stockout rates in the filtered telemetry.')

In [ ]:
# ============================================================
# Q4: BATTERY & EQUIPMENT PERFORMANCE
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Q4: Battery Health & Equipment Performance', fontsize=15, fontweight='bold',
             color='#e8eaf6', y=1.02)

# SOH distribution by supplier
ax = axes[0]
for i, (sup, color) in enumerate(zip(['Cellora','Amptek','Kyron'],
                                       ['#4ff7a0','#4f8ef7','#f74f4f'])):
    data = batteries[batteries['supplier']==sup]['current_soh_pct']
    ax.hist(data, bins=30, alpha=0.7, color=color, label=f'{sup} (n={len(data):,})', density=True)
ax.set_xlabel('Current SOH %'); ax.set_ylabel('Density')
ax.set_title('Battery SOH Distribution by Supplier')
ax.axvline(x=70, color='white', linestyle='--', alpha=0.5, label='70% SOH threshold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Kyron lot breakdown
ax = axes[1]
kyron = batteries[batteries['supplier']=='Kyron']
lot_soh = kyron.groupby('manufacturing_lot')['current_soh_pct'].mean().sort_values()
bar_colors = ['#f74f4f' if v < 70 else '#4ff7a0' for v in lot_soh.values]
ax.barh(lot_soh.index, lot_soh.values, color=bar_colors)
ax.set_xlabel('Mean Current SOH %')
ax.set_title('Kyron Manufacturing Lots: Mean Current SOH')
ax.axvline(x=80, color='white', linestyle='--', alpha=0.5, label='Cellora/Amptek baseline')
ax.legend(fontsize=9); ax.grid(True, axis='x', alpha=0.3)
# Annotate
ax.annotate('KY-2407/08/09\nDEFECT LOTS\n(1,461 packs)', 
            xy=(61, 1), fontsize=9, color='#f74f4f',
            bbox=dict(boxstyle='round', facecolor='rgba(247,79,79,0.1)', edgecolor='#f74f4f'))

# Cabinet temp vs charge time
ax = axes[2]
temp_charge = ok[['cabinet_temp_c','avg_charge_minutes']].dropna().sample(10000, random_state=42)
sc = ax.scatter(temp_charge['cabinet_temp_c'], temp_charge['avg_charge_minutes'],
                alpha=0.1, c=temp_charge['avg_charge_minutes'], cmap='YlOrRd', s=5)
ax.set_xlabel('Cabinet Temperature (°C)')
ax.set_ylabel('Charge Time (minutes)')
ax.set_title('Cabinet Temperature vs Charge Time')
ax.axvline(x=45, color='#f74f4f', linestyle='--', label='>45°C: charge time doubles')
plt.colorbar(sc, ax=ax, label='Charge time (min)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/q4_battery_performance.png', dpi=150, bbox_inches='tight',
            facecolor='#1a1d27')
plt.show()

print('KEY FINDING: Kyron KY-2407/08/09 show 38% degradation vs expected 18%.')
print(f'Mean SOH: Cellora={batteries[batteries.supplier=="Cellora"]["current_soh_pct"].mean():.1f}% '
      f'Amptek={batteries[batteries.supplier=="Amptek"]["current_soh_pct"].mean():.1f}% '
      f'Kyron={batteries[batteries.supplier=="Kyron"]["current_soh_pct"].mean():.1f}%')
print('The temperature/charge-time plot shows an association; it does not by itself prove causation.')

In [ ]:
# ============================================================
# Q5: PRICING & PARTNER ECONOMICS
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Q5: Partner Economics & Pricing Impact', fontsize=15, fontweight='bold',
             color='#e8eaf6', y=1.02)

# Partner discount vs rider count
ax = axes[0]
fleet_riders = riders[riders['partner_id'].notna()].groupby('partner_id').size().reset_index(name='rider_count')
fleet_merged = fleet.merge(fleet_riders, on='partner_id', how='left')
fleet_merged['effective_discount'] = fleet_merged['discount_pct_after_amendment'].fillna(fleet_merged['discount_pct'])
fleet_merged = fleet_merged.sort_values('rider_count', ascending=False).head(8)

bar_colors = ['#f74f4f' if p == 'FP-03' else '#4f8ef7' for p in fleet_merged['partner_id']]
bars = ax.bar(fleet_merged['partner_name'], fleet_merged['rider_count'], color=bar_colors, alpha=0.8)
ax2 = ax.twinx()
ax2.plot(range(len(fleet_merged)), fleet_merged['effective_discount'], 'D--', 
         color='#f7c94f', linewidth=2, markersize=8)
ax.set_xticklabels(fleet_merged['partner_name'], rotation=35, ha='right')
ax.set_ylabel('Rider Count', color='#9ca3c0')
ax2.set_ylabel('Effective Discount %', color='#f7c94f')
ax.set_title('Fleet Partner: Riders vs Discount (Red = ZipDrop)')
# Annotate ZipDrop
ax.annotate('ZipDrop:\n12%→28%\n(Nov 2024)', xy=(1, 2381), xytext=(3, 2200),
            fontsize=9, color='#f74f4f',
            arrowprops=dict(arrowstyle='->', color='#f74f4f'))
ax.grid(True, axis='y', alpha=0.3)

# Monthly ticket categories trend
ax = axes[1]
cat_monthly = tickets.groupby(['month','category']).size().unstack(fill_value=0)
cat_monthly = cat_monthly[(cat_monthly.index >= '2024-01') & (cat_monthly.index <= '2025-06')]
months_range = [str(m) for m in cat_monthly.index]
x = range(len(months_range))
for cat, color in zip(['no_battery_available','low_range','long_queue'],
                       ['#f74f4f','#f7c94f','#f97316']):
    if cat in cat_monthly.columns:
        ax.plot(x, cat_monthly[cat], color=color, linewidth=2, label=cat.replace('_',' ').title())
ax.set_title('Key Complaint Categories Over Time')
ax.set_xticks(list(x)[::3])
ax.set_xticklabels(months_range[::3], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Tickets / month')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
# Annotate Kyron onset
ax.axvline(x=9, color='#bf4ff7', linestyle='--', alpha=0.7, label='Kyron commissioned')

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/q5_pricing_partners.png', dpi=150, bbox_inches='tight',
            facecolor='#1a1d27')
plt.show()

print('KEY FINDING: ZipDrop contract amended Nov 2024: 12%→28% discount + peak surcharge waived.')
print('The contract change warrants review; exact margin impact requires a complete cost model.')
print('Low_range tickets closely track Kyron commissioning timeline (Sep-Nov 2024).')

### Q6 interpretation limitation

This section measures rider support-ticket exposure and service-issue patterns. It does **not** calculate confirmed churn because a support ticket is not evidence that a rider stopped using the service. Describe the results as retention-risk indicators unless a rider-level inactivity/return-use definition is separately implemented.


In [ ]:
# ============================================================
# Q6: RIDER EXPERIENCE — RETENTION RISK INDICATORS
# ============================================================

# Rider-ticket linkage
rider_ticket_summary = tickets.groupby('rider_id').agg(
    num_tickets=('ticket_id','count'),
    categories=('category', lambda x: x.tolist()),
    first_ticket=('created_ts','min')
).reset_index()

riders_enriched = riders.merge(rider_ticket_summary, on='rider_id', how='left')
riders_enriched['has_ticket'] = riders_enriched['num_tickets'].notna()
riders_enriched['ticket_count'] = riders_enriched['num_tickets'].fillna(0)
riders_2024plus = riders_enriched[riders_enriched['signup_date'] >= '2024-01-01']

print('=== RIDER EXPERIENCE / RETENTION RISK INDICATORS ===')
print(f'Riders signed up 2024+: {len(riders_2024plus):,}')
print(f'Of these, raised at least 1 ticket: {riders_2024plus["has_ticket"].sum():,} ({riders_2024plus["has_ticket"].mean()*100:.1f}%)')

print('\nSign-up channel vs ticket rate:')
print(riders_2024plus.groupby('signup_channel')['has_ticket'].mean().sort_values(ascending=False))

print('\nCity vs ticket rate:')
print(riders_2024plus.groupby('city')['has_ticket'].mean().sort_values(ascending=False))

# Gen1 ticket concentration
gen1_stations_list = stations[stations['charger_generation']=='Gen1']['station_id'].tolist()
gen1_tickets = tickets[tickets['station_id'].isin(gen1_stations_list)]
gen1_no_batt = gen1_tickets[gen1_tickets['category']=='no_battery_available']
total_no_batt = tickets[tickets['category']=='no_battery_available']
print(f'\nGen1 share of no_battery_available tickets: {len(gen1_no_batt)/len(total_no_batt)*100:.1f}%')
print(f'Gen1 share of stations: {len(gen1_stations_list)}/{len(stations)} = {len(gen1_stations_list)/len(stations)*100:.1f}%')

# Visualize retention risk factors
fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle('Q6: Rider Retention — Root Cause Factor Analysis', fontsize=15, 
             fontweight='bold', color='#e8eaf6')

cat_monthly = tickets.groupby(['month','category']).size().unstack(fill_value=0)
cat_monthly = cat_monthly[(cat_monthly.index >= '2024-01') & (cat_monthly.index <= '2025-06')]
months_range = [str(m) for m in cat_monthly.index]
x = range(len(months_range))

cat_colors = {'no_battery_available':'#f74f4f','low_range':'#f7c94f',
              'long_queue':'#f97316','app_issue':'#4f8ef7','billing_dispute':'#4ff7a0'}
for cat, color in cat_colors.items():
    if cat in cat_monthly.columns:
        ax.fill_between(x, cat_monthly[cat], alpha=0.4, color=color, label=cat.replace('_',' ').title())
        ax.plot(x, cat_monthly[cat], color=color, linewidth=1.5)

ax.set_xticks(list(x)[::2])
ax.set_xticklabels(months_range[::2], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Monthly Ticket Count')
ax.set_title('All Ticket Categories — Trend (Primary=Red/Yellow, Secondary=Blue/Green)')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
# Mark key events
ax.axvline(x=9, color='#bf4ff7', linestyle='--', linewidth=1.5, alpha=0.8)
ax.text(9.3, ax.get_ylim()[1]*0.9, 'Kyron\ncommissioned', color='#bf4ff7', fontsize=8)
ax.axvline(x=10, color='#f74f4f', linestyle=':', linewidth=1.5, alpha=0.8)
ax.text(10.3, ax.get_ylim()[1]*0.75, 'ZipDrop\namendment', color='#f74f4f', fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/q6_retention.png', dpi=150, bbox_inches='tight',
            facecolor='#1a1d27')
plt.show()

print('\nKEY FINDING: Gen1 stations generate 56% of no_battery_available tickets despite being 34% of stations.')
print('Low-range ticket growth is an observed complaint trend; temporal association does not prove battery-lot causation.')
print('These are potential retention risks; ticket data does not confirm rider churn or causal drivers.')
print('Secondary: peak-hour queues (correlated with Gen1), billing confusion (pricing pilot).')

In [ ]:
# ============================================================
# SECTION 3: SUMMARY TABLE
# ============================================================

print('=' * 65)
print('VOLTRELAY ENERGY — ANALYTICS SUMMARY')
print('=' * 65)

print('\n📊 NETWORK OVERVIEW')
print(f'  Total registered riders: {len(riders):,}')
print(f'  Fleet-affiliated: {riders["partner_id"].notna().sum():,} ({riders["partner_id"].notna().mean()*100:.0f}%)')
print(f'  Active stations: {len(stations):,} across 6 cities')
print(f'  Station hourly observations: {len(shs):,}')
print(f'  Total support tickets: {len(tickets):,}')

print('\n⚡ SERVICE QUALITY')
so_rate = ok['stockout_2w'].mean()*100
print(f'  Overall 2W stockout rate: {so_rate:.2f}%')
print(f'  Peak hour (3pm) stockout rate: {ok[ok.hour==15]["stockout_2w"].mean()*100:.2f}%')
print(f'  Gen1 stockout rate: {ok[ok.charger_generation=="Gen1"]["stockout_2w"].mean()*100:.2f}%')
print(f'  Gen3 stockout rate: {ok[ok.charger_generation=="Gen3"]["stockout_2w"].mean()*100:.2f}%')

print('\n🔋 BATTERY HEALTH')
for sup in ['Cellora','Amptek','Kyron']:
    b = batteries[batteries['supplier']==sup]
    print(f'  {sup}: {len(b):,} packs, mean SOH = {b["current_soh_pct"].mean():.1f}%, "\
          f"mean degradation = {b["soh_degradation"].mean():.1f}%')
kyron_bad = batteries[(batteries['supplier']=='Kyron') & 
                       (batteries['manufacturing_lot'].isin(['KY-2407','KY-2408','KY-2409']))]
print(f'  Kyron defect lots (KY-2407/08/09): {len(kyron_bad):,} packs = {len(kyron_bad)/len(batteries)*100:.1f}% of fleet')

print('\n📈 TICKET TREND')
jan24_t = tickets[tickets['month']=='2024-01'].shape[0]
jun25_t = tickets[tickets['month']=='2025-06'].shape[0]
print(f'  Jan 2024: {jan24_t:,} tickets')
print(f'  Jun 2025: {jun25_t:,} tickets')
print(f'  Growth: +{(jun25_t/jan24_t-1)*100:.0f}%')

print('\n💸 KEY PRICING EVENT')
print('  ZipDrop (FP-03): discount amended Nov 2024 from 12% → 28%')
print('  Peak surcharge billing is waived under the recorded ZipDrop contract terms.')

print('\n🎯 TOP 3 RECOMMENDATIONS')
print('  1. [IMMEDIATE] Quarantine Kyron KY-2407/08/09 batteries (1,461 packs at 61% SOH)')
print('  2. [SHORT-TERM] Prioritise Gen1 station upgrades using verified station-level performance and location data')
print('  3. [REVIEW] Evaluate ZipDrop contract economics using partner-level costs and tariff rules')
print('=' * 65)